In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
"""
Experiment 3: Barrier Opacity vs Symbol Quality
================================================

Nomura (2025) found that decentralized constraints — agents cannot read each
other's internal states — produce *better* symbol systems than brain-connected
agents that share everything. They only tested two conditions: fully connected
vs fully decentralized.

Our hypothesis (from the Deutsch insight about discreteness and error correction):
    The relationship between opacity and symbol quality is NON-MONOTONIC.
    There is an optimal opacity level. Too transparent: agents use continuous
    internal activations and never develop meaningful symbols. Too opaque: agents
    can't coordinate. The sweet spot is where the channel forces discretization
    but still allows enough information through.

How we sweep opacity:
    We parameterize opacity via Gumbel-Softmax temperature τ:
      τ → 0   : hard discrete (fully opaque, each agent gets a token)
      τ → ∞   : soft continuous (fully transparent, full gradient flows)
    We also add a noise injection level σ on top of the discrete channel
    to get intermediate regimes beyond what temperature alone provides.

    The full opacity axis: [(τ=0.1, σ=0), (τ=0.3, σ=0), (τ=1.0, σ=0),
                            (τ=2.0, σ=0), (τ=5.0, σ=0), continuous baseline]

Symbol quality metric: Representational Similarity Analysis (RSA)
    - Compute pairwise distances between all input objects in meaning space
    - Compute pairwise distances between corresponding messages in symbol space
    - RSA score = Spearman correlation between the two distance matrices
    - High RSA: the symbol space faithfully mirrors the meaning space
    - This is the same metric used by Nomura et al. and by Lazaridou et al.

Expected result (our hypothesis):
    RSA peaks at intermediate τ, drops at both extremes.
    If the curve is monotonic instead (higher opacity always better), that
    partially confirms Nomura but not the non-monotonic Deutsch argument.
    Either way it's a publishable empirical sweep.

Kaggle setup:
    !git clone https://github.com/facebookresearch/EGG.git
    !pip install -e EGG
    !pip install scipy matplotlib seaborn
"""

import os
import json
import argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from scipy.stats import spearmanr
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


# ---------------------------------------------------------------------------
# Data
# ---------------------------------------------------------------------------

def make_dataset(n_features, n_distractors, n_samples, seed=42):
    rng = np.random.default_rng(seed)
    objects = rng.integers(0, 2,
        size=(n_samples, n_distractors + 1, n_features)).astype(np.float32)
    sender_input   = torch.tensor(objects[:, 0, :])
    labels         = torch.zeros(n_samples, dtype=torch.long)
    receiver_input = torch.tensor(objects)
    return TensorDataset(sender_input, labels, receiver_input)


# ---------------------------------------------------------------------------
# Agents
# ---------------------------------------------------------------------------

class Sender(nn.Module):
    def __init__(self, n_features, hidden, vocab_size):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),     nn.ReLU(),
            nn.Linear(hidden, vocab_size),
        )

    def forward(self, x, _aux=None):
        return self.fc(x)


class Receiver(nn.Module):
    def __init__(self, n_features, hidden, msg_dim, n_distractors):
        super().__init__()
        self.msg_embed = nn.Linear(msg_dim, hidden)
        self.obj_embed = nn.Linear(n_features, hidden)

    def forward(self, msg, _input, receiver_input, _aux=None):
        # receiver_input: (B, n_candidates, n_features)
        msg_h  = self.msg_embed(msg)                              # (B, hidden)
        obj_h  = self.obj_embed(receiver_input)                   # (B, n_cand, hidden)
        scores = torch.bmm(obj_h, msg_h.unsqueeze(-1)).squeeze(-1)
        return scores


# ---------------------------------------------------------------------------
# Parameterized opacity channel
# ---------------------------------------------------------------------------

class OpacityGame(nn.Module):
    """
    Referential game where channel opacity is controlled by:
      - gs_temperature (τ): lower = more discrete
      - channel_noise (σ):  additive noise after discretization

    opacity_mode:
      'gumbel'     : Gumbel-Softmax at temperature τ (our main sweep)
      'continuous' : raw sender output, no discretization (fully transparent)
      'hard'       : argmax + one-hot, no gradients (fully opaque, eval only)
    """
    def __init__(self, sender, receiver, vocab_size,
                 gs_temperature=1.0, channel_noise=0.0,
                 opacity_mode='gumbel'):
        super().__init__()
        self.sender        = sender
        self.receiver      = receiver
        self.vocab_size    = vocab_size
        self.temperature   = gs_temperature
        self.channel_noise = channel_noise
        self.opacity_mode  = opacity_mode

    def encode(self, logits):
        """Convert sender logits to message according to opacity mode."""
        if self.opacity_mode == 'continuous':
            # Raw logits passed through — fully transparent, gradient flows everywhere
            msg = torch.softmax(logits, dim=-1)
        elif self.opacity_mode == 'hard':
            # Hard argmax, no gradient (evaluation only)
            idx = logits.argmax(dim=-1)
            msg = F.one_hot(idx, self.vocab_size).float()
        else:
            # Gumbel-Softmax: tunable between continuous (high τ) and discrete (low τ)
            if self.training:
                msg = F.gumbel_softmax(logits, tau=self.temperature, hard=True)
            else:
                idx = logits.argmax(dim=-1)
                msg = F.one_hot(idx, self.vocab_size).float()

        # Optionally add channel noise (models partial opacity beyond GS)
        if self.channel_noise > 0 and self.training:
            msg = msg + torch.randn_like(msg) * self.channel_noise
            # re-normalise to keep it on simplex
            msg = F.softmax(msg / (self.temperature + 1e-8), dim=-1)

        return msg

    def forward(self, sender_input, labels, receiver_input):
        logits = self.sender(sender_input)
        msg    = self.encode(logits)
        scores = self.receiver(msg, None, receiver_input)
        loss   = F.cross_entropy(scores, labels)
        acc    = (scores.argmax(-1) == labels).float().mean()
        return loss, {"acc": acc}, logits, msg


# ---------------------------------------------------------------------------
# Training loop (standalone, no EGG dependency for this experiment)
# ---------------------------------------------------------------------------

def train_game(game, train_ds, val_ds, args):
    optimizer    = torch.optim.Adam(game.parameters(), lr=args.lr)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=256)

    best_val_acc = 0.0
    for epoch in range(args.n_epochs):
        game.train()
        for si, lb, ri in train_loader:
            optimizer.zero_grad()
            loss, info, _, _ = game(si, lb, ri)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 20 == 0:
            game.eval()
            val_accs = []
            with torch.no_grad():
                for si, lb, ri in val_loader:
                    _, info, _, _ = game(si, lb, ri)
                    val_accs.append(info["acc"].item())
            val_acc = float(np.mean(val_accs))
            best_val_acc = max(best_val_acc, val_acc)
            print(f"  epoch {epoch+1:3d} | val_acc={val_acc:.4f}")

    return best_val_acc


# ---------------------------------------------------------------------------
# RSA: Representational Similarity Analysis
# ---------------------------------------------------------------------------

def compute_rsa(game, dataset, n_samples=500):
    """
    RSA score: Spearman correlation between:
      - pairwise Hamming distances in *meaning space* (input objects)
      - pairwise distances in *message space* (what agents actually send)

    A high RSA score means the symbol system is a faithful map of the
    structure of the world — the geometry of meanings is preserved in symbols.

    This is the same metric as in Nomura et al. and Lazaridou et al. (2018).
    """
    game.eval()
    loader = DataLoader(dataset, batch_size=n_samples, shuffle=True)
    si, lb, ri = next(iter(loader))

    with torch.no_grad():
        logits = game.sender(si)
        # For RSA we always use hard messages (eval mode)
        idx  = logits.argmax(dim=-1)
        msgs = F.one_hot(idx, game.vocab_size).float().numpy()   # (N, vocab)

    meanings = si.numpy()   # (N, n_features)

    # Pairwise distances
    meaning_dists = pdist(meanings, metric='hamming')
    message_dists = pdist(msgs,     metric='hamming')

    rho, pval = spearmanr(meaning_dists, message_dists)
    return float(rho), float(pval)


def compute_entropy(game, dataset, n_samples=500):
    """
    Average entropy of sender's symbol distribution.
    Low entropy = sender is confidently using specific symbols.
    High entropy = sender is hedging / not committed to symbols.
    """
    game.eval()
    loader = DataLoader(dataset, batch_size=n_samples, shuffle=True)
    si, lb, ri = next(iter(loader))
    with torch.no_grad():
        logits = game.sender(si)
        probs  = F.softmax(logits, dim=-1)
        ent    = -(probs * (probs + 1e-9).log()).sum(dim=-1).mean()
    return float(ent)


def compute_context_independence(game, dataset, n_samples=1000):
    """
    Context independence: how consistently does the same meaning get the
    same symbol across different distractor sets?
    High CI = stable, object-level symbols.
    Low CI  = context-dependent signals (not true symbols).

    We measure this by sampling the same target objects with different
    distractor sets and checking if they get the same token.
    (RSA measures structure; CI measures stability — complementary.)
    """
    game.eval()
    rng = np.random.default_rng(123)
    n_features    = dataset[0][0].shape[0]
    n_distractors = dataset[0][2].shape[0] - 1

    # Sample unique target objects
    targets = torch.tensor(
        rng.integers(0, 2, size=(n_samples, n_features)).astype(np.float32)
    )

    assignments = []
    for _ in range(3):   # three different distractor contexts
        distractors = torch.tensor(
            rng.integers(0, 2,
                size=(n_samples, n_distractors, n_features)).astype(np.float32)
        )
        with torch.no_grad():
            logits = game.sender(targets)
            tokens = logits.argmax(dim=-1).numpy()
        assignments.append(tokens)

    # CI = fraction of samples where all 3 contexts gave the same token
    assignments = np.stack(assignments, axis=1)  # (N, 3)
    ci = float((assignments[:, 0] == assignments[:, 1]).mean() * \
               (assignments[:, 1] == assignments[:, 2]).mean())
    return ci


# ---------------------------------------------------------------------------
# Main sweep
# ---------------------------------------------------------------------------

def run_opacity_sweep(args):
    os.makedirs(args.results_dir, exist_ok=True)

    train_ds = make_dataset(args.n_features, args.n_distractors, args.n_train, args.seed)
    val_ds   = make_dataset(args.n_features, args.n_distractors, args.n_val,   args.seed + 1)

    # Define opacity conditions
    # Each entry: (label, gs_temperature, channel_noise, opacity_mode)
    # Ordered from most transparent to most opaque
    conditions = [
        ("continuous",   99.0, 0.0, "continuous"),   # fully transparent
        ("gs_tau=10",    10.0, 0.0, "gumbel"),
        ("gs_tau=5",      5.0, 0.0, "gumbel"),
        ("gs_tau=2",      2.0, 0.0, "gumbel"),
        ("gs_tau=1",      1.0, 0.0, "gumbel"),
        ("gs_tau=0.5",    0.5, 0.0, "gumbel"),
        ("gs_tau=0.3",    0.3, 0.0, "gumbel"),
        ("gs_tau=0.1",    0.1, 0.0, "gumbel"),       # most opaque / most discrete
    ]

    results = []

    for label, tau, sigma, mode in conditions:
        print(f"\n{'='*55}")
        print(f"Condition: {label}  (τ={tau}, σ={sigma}, mode={mode})")
        print(f"{'='*55}")

        sender   = Sender(args.n_features, args.hidden, args.vocab_size)
        receiver = Receiver(args.n_features, args.hidden, args.vocab_size,
                            args.n_distractors)
        game     = OpacityGame(sender, receiver, args.vocab_size,
                               gs_temperature=tau,
                               channel_noise=sigma,
                               opacity_mode=mode)

        val_acc = train_game(game, train_ds, val_ds, args)

        rsa_score, rsa_pval = compute_rsa(game, val_ds)
        entropy             = compute_entropy(game, val_ds)
        ci                  = compute_context_independence(game, val_ds)

        row = {
            "condition"    : label,
            "temperature"  : tau,
            "channel_noise": sigma,
            "opacity_mode" : mode,
            "val_acc"      : round(val_acc,    4),
            "rsa_score"    : round(rsa_score,  5),
            "rsa_pval"     : round(rsa_pval,   6),
            "entropy"      : round(entropy,    4),
            "context_ind"  : round(ci,         4),
        }
        results.append(row)
        print(json.dumps(row))

    # Save
    out_path = os.path.join(args.results_dir, "opacity_sweep.json")
    with open(out_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nResults saved to {out_path}")

    return results


# ---------------------------------------------------------------------------
# Plotting
# ---------------------------------------------------------------------------

def plot_results(results, save_dir):
    labels    = [r["condition"]   for r in results]
    rsa       = [r["rsa_score"]   for r in results]
    acc       = [r["val_acc"]     for r in results]
    entropy   = [r["entropy"]     for r in results]
    ci        = [r["context_ind"] for r in results]
    x         = np.arange(len(labels))

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        "Barrier Opacity vs Symbol Quality\n"
        "Left = transparent (continuous), Right = opaque (discrete)",
        fontsize=14, fontweight='bold'
    )

    def bar_plot(ax, values, title, ylabel, color):
        bars = ax.bar(x, values, color=color, alpha=0.8, edgecolor='black', linewidth=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
        ax.set_title(title, fontsize=11)
        ax.set_ylabel(ylabel, fontsize=10)
        # highlight the peak
        peak_idx = int(np.argmax(values))
        bars[peak_idx].set_edgecolor('red')
        bars[peak_idx].set_linewidth(2.5)
        ax.axhline(y=values[peak_idx], color='red', linestyle='--', alpha=0.4, linewidth=1)

    bar_plot(axes[0, 0], rsa,
             "RSA Score\n(symbol–meaning alignment)",
             "Spearman ρ", "#1976D2")

    bar_plot(axes[0, 1], acc,
             "Task Accuracy",
             "Accuracy", "#388E3C")

    bar_plot(axes[1, 0], entropy,
             "Sender Symbol Entropy\n(lower = more committed symbols)",
             "Entropy (nats)", "#F57C00")

    bar_plot(axes[1, 1], ci,
             "Context Independence\n(higher = more stable symbols)",
             "CI score", "#7B1FA2")

    plt.tight_layout()
    path = os.path.join(save_dir, "opacity_sweep.png")
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to {path}")

    # Second plot: RSA and accuracy overlaid on same axis to show tradeoff
    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax2 = ax1.twinx()

    ax1.plot(x, rsa, 'o-', color="#1976D2", linewidth=2, markersize=8, label="RSA score")
    ax2.plot(x, acc, 's--', color="#388E3C", linewidth=2, markersize=8, label="Task accuracy")

    ax1.set_xlabel("Opacity condition (left=transparent, right=discrete)", fontsize=11)
    ax1.set_ylabel("RSA score (symbol quality)", color="#1976D2", fontsize=11)
    ax2.set_ylabel("Task accuracy",               color="#388E3C", fontsize=11)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower left')

    plt.title("RSA Symbol Quality vs Task Accuracy across Opacity Levels\n"
              "Non-monotonic RSA = Deutsch argument confirmed", fontsize=12)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "rsa_vs_accuracy.png")
    plt.savefig(path2, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Plot saved to {path2}")


# ---------------------------------------------------------------------------
# Args + entry point
# ---------------------------------------------------------------------------

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--n_features",    type=int,   default=16)
    p.add_argument("--n_distractors", type=int,   default=9)
    p.add_argument("--n_train",       type=int,   default=10000)
    p.add_argument("--n_val",         type=int,   default=2000)
    p.add_argument("--vocab_size",    type=int,   default=32)
    p.add_argument("--hidden",        type=int,   default=64)
    p.add_argument("--n_epochs",      type=int,   default=80)
    p.add_argument("--batch_size",    type=int,   default=128)
    p.add_argument("--lr",            type=float, default=1e-3)
    p.add_argument("--results_dir",   type=str,   default="results")
    p.add_argument("--seed",          type=int,   default=42)
    return p.parse_args()


def main():
    args    = parse_args()
    torch.manual_seed(args.seed)

    results = run_opacity_sweep(args)
    plot_results(results, args.results_dir)

    # Print summary interpretation
    rsa_scores = [r["rsa_score"] for r in results]
    peak_idx   = int(np.argmax(rsa_scores))
    peak_cond  = results[peak_idx]["condition"]

    print("\n=== SUMMARY ===")
    print(f"Peak RSA at condition: {peak_cond}")
    if 0 < peak_idx < len(results) - 1:
        print("→ NON-MONOTONIC: RSA peaks at intermediate opacity.")
        print("  This confirms the Deutsch hypothesis: optimal barrier, not maximum.")
    elif peak_idx == len(results) - 1:
        print("→ MONOTONIC: RSA keeps rising with opacity.")
        print("  This confirms Nomura but not the non-monotonic prediction.")
    else:
        print("→ TRANSPARENCY WINS: continuous performs best.")
        print("  Consider increasing n_epochs or vocab_size.")


if __name__ == "__main__":
    main()